## 미리보기

In [ ]:
# KoNLPy는 내부적으로 Java를 사용합니다
# Java를 먼저 설치해야 KoNLPy가 작동합니다
!apt-get install default-jdk -y

# konlpy : 한국어 형태소 분석 라이브러리
# nltk   : 영어 자연어 처리 라이브러리
# --quiet : 설치 메시지를 간략하게 출력
!pip install konlpy nltk --quiet

# 설치 완료 메시지
print("설치 완료!")

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  at-spi2-core default-jdk-headless default-jre default-jre-headless
  fonts-dejavu-core fonts-dejavu-extra gsettings-desktop-schemas
  libatk-bridge2.0-0 libatk-wrapper-java libatk-wrapper-java-jni libatk1.0-0
  libatk1.0-data libatspi2.0-0 libxcomposite1 libxt-dev libxtst6 libxxf86dga1
  openjdk-11-jdk openjdk-11-jdk-headless openjdk-11-jre
  openjdk-11-jre-headless session-migration x11-utils
Suggested packages:
  libxt-doc openjdk-11-demo openjdk-11-source visualvm libnss-mdns
  fonts-ipafont-gothic fonts-ipafont-mincho fonts-wqy-microhei
  | fonts-wqy-zenhei fonts-indic mesa-utils
The following NEW packages will be installed:
  at-spi2-core default-jdk default-jdk-headless default-jre
  default-jre-headless fonts-dejavu-core fonts-dejavu-extra
  gsettings-desktop-schemas libatk-bridge2.0-0 libatk-wrapper-java
  libatk-wrapper-java-jn

In [ ]:
import re                    # 정규표현식 : 텍스트에서 특정 패턴을 찾거나 바꿀 때 사용
from collections import Counter  # Counter : 단어 빈도를 쉽게 셀 때 사용

import nltk                             # 영어 자연어 처리 도구 모음
nltk.download('punkt',     quiet=True)  # 문장/단어 토큰화 모델
nltk.download('punkt_tab', quiet=True)  # 토큰화 추가 데이터
nltk.download('stopwords', quiet=True)  # 영어 불용어 목록
nltk.download('wordnet',   quiet=True)  # 영어 단어 원형 데이터

from konlpy.tag import Okt   # Okt : KoNLPy에서 가장 많이 쓰이는 한국어 형태소 분석기
okt = Okt()                  # 형태소 분석기 객체 생성 (한 번만 만들면 계속 사용 가능)

print(" 준비 완료!")

 준비 완료!


In [ ]:
리뷰1 = "ㅋㅋㅋ 진짜 맛있어요!!! 근데 배송이 너무 느리네요ㅠㅠ"

단어들1 = 리뷰1.split()

print("원본 리뷰: ", 리뷰1)
print("split() 결과: ", 단어들1)
print("단어 수: ", len(단어들1), "개")

원본 리뷰:  ㅋㅋㅋ 진짜 맛있어요!!! 근데 배송이 너무 느리네요ㅠㅠ
split() 결과:  ['ㅋㅋㅋ', '진짜', '맛있어요!!!', '근데', '배송이', '너무', '느리네요ㅠㅠ']
단어 수:  7 개


In [ ]:
리뷰2 = "<b>최고</b>의 치킨!! http://naver.com 강추★★★ 가격도 저렴👍"

단어들2 = 리뷰2.split()

print("원본 리뷰: ", 리뷰2)
print("split() 결과: ", 단어들2)
print("단어 수: ", len(단어들2), "개")

원본 리뷰:  <b>최고</b>의 치킨!! http://naver.com 강추★★★ 가격도 저렴👍
split() 결과:  ['<b>최고</b>의', '치킨!!', 'http://naver.com', '강추★★★', '가격도', '저렴👍']
단어 수:  6 개


In [ ]:
리뷰3a = "맛잇다"   # 오탈자
리뷰3b = "맛있어요"  # 격식체
리뷰3c = "맛있다"    # 기본형

print("컴퓨터 입장에서 a == b:", 리뷰3a == 리뷰3b)  # False!
print("컴퓨터 입장에서 b == c:", 리뷰3b == 리뷰3c)  # False!

컴퓨터 입장에서 a == b: False
컴퓨터 입장에서 b == c: False


## 0. 전처리 4단계 흐름

전처리는 다음 4단계 순서로 진행합니다

```
원본  →  1️⃣ 정제  →  2️⃣ 정규화  →  3️⃣ 토큰화  →  4️⃣ 불용어 제거  →  완료!
```


In [ ]:
원본 = "ㅋㅋㅋ 진짜 맛있어요!!!! 배달이 너무 느려요ㅠㅠ http://shop.kr"

print("원본 텍스트: ", 원본)

원본 텍스트:  ㅋㅋㅋ 진짜 맛있어요!!!! 배달이 너무 느려요ㅠㅠ http://shop.kr


### 1. 정제 (노이즈 제거 - 특수문자, html태그 등)
- **목표: URL, 특수문자 등 쓸모없는 것들을 제거**
- re.sub(패턴, 바꿀문자, 원본): 패턴과 일치하는 부분을 바꿀 문자로 교체
  - r'http\S+' : http로 시작하는 URL 패턴
  - r'[^가-힣a-zA-Z0-9 ]' : 한글·영문·숫자·공백이 아닌 것을 모두 선택
  - r'\s+' : 공백이 2개 이상 연속된 부분
  - .strip() : 앞뒤 공백 제거

In [ ]:
step1 = re.sub(r'http\S+', "", 원본)
step1 = re.sub(r'[^가-힣a-zA-Z0-9 ]', "", step1)
step1 = re.sub(r'\s+', " ", step1).strip()

print(f"정제 결과: {원본} -> {step1}")

정제 결과: ㅋㅋㅋ 진짜 맛있어요!!!! 배달이 너무 느려요ㅠㅠ http://shop.kr -> 진짜 맛있어요 배달이 너무 느려요


In [ ]:
print(len(step1))

18


### 2. 정규화 단계
- **목표: 반복되는 표현을 통일**
- r'(.)\1{2,}' : 같은 문자가 3번 이상 반복되는 패턴
  - (.)   : 아무 문자 1개
  - \1    : 바로 앞에서 잡은 그 문자와 동일한 문자
  - {2,}  : 2번 이상 더 반복 → 합치면 3번 이상 반복
  - r'\1\1' : 그 문자를 2번으로 줄임

In [ ]:
step2 = re.sub(r'(.)\1{2,}', r'\1\1', step1)

print(f"정규화 결과: {원본} -> {step2}")

정규화 결과: ㅋㅋㅋ 진짜 맛있어요!!!! 배달이 너무 느려요ㅠㅠ http://shop.kr -> 진짜 맛있어요 배달이 너무 느려요


### 3. 토큰화 단계
- **목표: 텍스트를 단어 단위로 쪼갠다**
- okt.nouns() : 한국어 문장에서 명사만 뽑아준다

In [ ]:
step3 = okt.nouns(step2)

print(f"토큰화 후: {원본} -> {step3}")

토큰화 후: ㅋㅋㅋ 진짜 맛있어요!!!! 배달이 너무 느려요ㅠㅠ http://shop.kr -> ['진짜', '배달']


### 4. 불용어 제거 단계
- **목표: 분석에 도움이 안되는 단어 제거**
- 리스트 컴프리헨션: 조건을 만족하는 단어만 남기는 방법
  - w not in 불용어: 불용어 목록에 없는 단어
  - len(w) >= 2: 2글자 이상인 단어만

In [ ]:
불용어 = ["것", "수", "나", "저", "그", "이", "좀", "더"]

step4 = [w for w in step3 if w not in 불용어 and len(w) >= 2]

print(f"불용어 제거 후: {원본} -> {step4}")

불용어 제거 후: ㅋㅋㅋ 진짜 맛있어요!!!! 배달이 너무 느려요ㅠㅠ http://shop.kr -> ['진짜', '배달']


## 1. (3단계) 토큰화(Tokenization)

### 1) **split()**: 공백 기준으로 토큰화
-> 한계: 특수문자가 붙어있을 시 제대로 분리 안 됨.

In [ ]:
문장 = "나는 오늘 학교에서 파이썬을 배웠어요"

토큰들 = 문장.split()

print("원본:", 문장)
print("결과:", 토큰들)
print("개수:", len(토큰들), "개")

# split() 한계
문장2 = "오늘 날씨가 정말 좋네요!!! 산책가고 싶다ㅠㅠ"

결과 = 문장2.split()

print("-------")
print("문장에 특수문자가 있을 경우")
print("원본:", 문장2)
print("결과:", 결과)

원본: 나는 오늘 학교에서 파이썬을 배웠어요
결과: ['나는', '오늘', '학교에서', '파이썬을', '배웠어요']
개수: 5 개
-------
문장에 특수문자가 있을 경우
원본: 오늘 날씨가 정말 좋네요!!! 산책가고 싶다ㅠㅠ
결과: ['오늘', '날씨가', '정말', '좋네요!!!', '산책가고', '싶다ㅠㅠ']


### 2) NLTK: **word_tokenize()** - 영어 전용
--> 문장 부호를 단어에서 자동으로 분리

In [ ]:
from nltk.tokenize import word_tokenize

영어문장 =  "I love text mining!! It's really amazing."

결과 = word_tokenize(영어문장)

print("원본:", 영어문장)
print("결과:", 결과)


원본: I love text mining!! It's really amazing.
결과: ['I', 'love', 'text', 'mining', '!', '!', 'It', "'s", 'really', 'amazing', '.']


### 3) NLTK **sent_tokenize()**
-> 단어가 아닌 **문장** 단위로 쪼갠다.

In [ ]:
from nltk.tokenize import sent_tokenize

긴텍스트 = "오늘 날씨가 좋네요. 산책을 나가고 싶어요. 하지만 할 일이 많습니다."

문장들 = sent_tokenize(긴텍스트)

print("원본:", 긴텍스트)
print("결과:", 문장들)

원본: 오늘 날씨가 좋네요. 산책을 나가고 싶어요. 하지만 할 일이 많습니다.
결과: ['오늘 날씨가 좋네요.', '산책을 나가고 싶어요.', '하지만 할 일이 많습니다.']


In [ ]:
print("문장 1:", 문장들[0])
print("문장 2:", 문장들[1])
print("문장 3:", 문장들[2])
print()
print("--> 총", len(문장들), "개의 문장으로 분리!")

문장 1: 오늘 날씨가 좋네요.
문장 2: 산책을 나가고 싶어요.
문장 3: 하지만 할 일이 많습니다.

--> 총 3 개의 문장으로 분리!


In [ ]:
나의_문장 = "파이썬 텍스트 분석이 생각보다 재미있어요"

# 방법 1: split()
방법1 = 나의_문장.split()
print("방법 1 - split():", 방법1)

# 방법 2: okt.morphs() — 형태소 분리 (CH 0에서 만든 okt 사용)
방법2 = okt.morphs(나의_문장)
print("방법 2 - morphs():", 방법2)

# 방법 3: okt.nouns() — 명사만 추출
방법3 = okt.nouns(나의_문장)
print("방법 3 - nouns():", 방법3)

방법 1 - split(): ['파이썬', '텍스트', '분석이', '생각보다', '재미있어요']
방법 2 - morphs(): ['파이썬', '텍스트', '분석', '이', '생각', '보다', '재미있어요']
방법 3 - nouns(): ['파이썬', '텍스트', '분석', '생각']


## 2. (1단계) 데이터 정제

### 1) 특수문자 제거
- re.sub(패턴, 바꿀것, 원본)
- r'[^가-힣 ]'  : 한글(가~힣)과 공백( ) 이외의 모든 문자

- r'[^가-힣a-zA-Z0-9 ]' :
- 가-힣  : 한글
- a-z    : 영어 소문자
- A-Z    : 영어 대문자
- 0-9    : 숫자
- ' '    : 공백
- ^ (맨앞): 위 목록에 없는 것 = 나머지 전부

In [ ]:
import re

텍스트1 = "ㅋㅋㅋ 진짜 맛있어요!!!! 최고★★★"

결과1 = re.sub(r'[^가-힣 ]', '', 텍스트1)

print("원본:", 텍스트1)
print("결과:", 결과1)

원본: ㅋㅋㅋ 진짜 맛있어요!!!! 최고★★★
결과:  진짜 맛있어요 최고


In [ ]:
텍스트2 = "Python 3.10은 정말 좋아요!! 버전 업그레이드★"

결과2 = re.sub(r'[^가-힣a-zA-Z0-9 ]', '', 텍스트2)

print("원본:", 텍스트2)
print("결과:", 결과2)

원본: Python 3.10은 정말 좋아요!! 버전 업그레이드★
결과: Python 310은 정말 좋아요 버전 업그레이드


### 2) URL 제거
- http: 'http' 로 시작하는
- \S+: 공백이 아닌 문자가 1개 이상 이어지는 것
-> http로 시작해서 공백 전까지의 모든 문자열 = URL

In [ ]:
텍스트3 = "이 사이트 강추해요! https://www.naver.com 여기서 구매했어요"

결과3 = re.sub(r'http\S+', '', 텍스트3)

# 결과에 연속 공백이 생겼을 수 있으므로 정리
결과3 = re.sub(r'\s+', ' ', 결과3).strip()

print("원본:", 텍스트3)
print("결과:", 결과3)

원본: 이 사이트 강추해요! https://www.naver.com 여기서 구매했어요
결과: 이 사이트 강추해요! 여기서 구매했어요


### 3) HTML 제거
- '<[^>]+>': <로 시작해서 >로 끝나는 패턴

In [ ]:
텍스트4 = "<h1>상품 후기</h1><p>이 상품은 <b>정말</b> 좋았어요!</p>"

결과4 = re.sub(r'<[^>]+>', '', 텍스트4)

print("원본:", 텍스트4)
print("결과:", 결과4)

원본: <h1>상품 후기</h1><p>이 상품은 <b>정말</b> 좋았어요!</p>
결과: 상품 후기이 상품은 정말 좋았어요!


### 4) 연속 공백 정리
- '\s+' : 공백 문자가 1개 이상 연속된 것

In [ ]:
텍스트5 = "안녕하세요   반갑습니다    오늘   날씨가    좋네요"

결과5 = re.sub(r'\s+', ' ', 텍스트5).strip()

# repr() : 공백이 눈에 보이도록 표시해주는 함수
print("원본:", repr(텍스트5))
print("결과:", repr(결과5))

원본: '안녕하세요   반갑습니다    오늘   날씨가    좋네요'
결과: '안녕하세요 반갑습니다 오늘 날씨가 좋네요'


In [ ]:
def 텍스트_정제(텍스트):
    텍스트 = re.sub(r'http\S+', '', 텍스트)             # 1. URL 제거
    텍스트 = re.sub(r'<[^>]+>', '', 텍스트)             # 2. HTML 태그 제거 (<...> 형태
    텍스트 = re.sub(r'[^가-힣a-zA-Z0-9 ]', '', 텍스트)   # 3. 특수문자 제거 (한글·영문·숫자·공백만 남김
    텍스트 = re.sub(r'\s+', ' ', 텍스트).strip()        # 4. 연속 공백을 공백 1개로 정리
    return 텍스트

In [ ]:
리뷰들 = [
    "ㅋㅋ 진짜 맛있어요!!!! http://naver.com 강추★",
    "<b>배송</b>이 너무 느려요 😤😤 별로임",
    "가격이   너무   비싸요...   다시는 안살듯"
]

for i in 리뷰들:
  정제된 = 텍스트_정제(i)
  print(f"원본: {i}")
  print(f"정제: {정제된}")
  print()

원본: ㅋㅋ 진짜 맛있어요!!!! http://naver.com 강추★
정제: 진짜 맛있어요 강추

원본: <b>배송</b>이 너무 느려요 😤😤 별로임
정제: 배송이 너무 느려요 별로임

원본: 가격이   너무   비싸요...   다시는 안살듯
정제: 가격이 너무 비싸요 다시는 안살듯



## 3. (2단계) 정규화
같은 의미를 가진 단어들을 하나의 형태로 통일하는 작업

### 1) 영어 대소문자 -> 소문자 통일
- .lower(): 문자열의 모든 영문자를 소문자로 바꾼다

In [ ]:
단어A = "Python"
단어B = "PYTHON"
단어C = "PyThOn"

print("소문자 변환 전:", 단어A, "|", 단어B, "|", 단어C)
print("소문자 변환 후:", 단어A.lower(), "|", 단어B.lower(), "|", 단어C.lower())

소문자 변환 전: Python | PYTHON | PyThOn
소문자 변환 후: python | python | python


### 2) 반복 문자 압축 함수 만들기
ㅋㅋㅋㅋ -> ㅋㅋ (의미는 살리되, 횟수 감소)
- r'(.)\1{2,}' : 같은 문자가 3번 이상 반복되는
패턴을 찾기
  - (.): 임의의 문자 1개를 그룹으로 캡처
  - \1: 앞에서 캡처한 문자와 동일한 문자
  - {2,}: 2번 이상 반복
- r'\1\1'      : 찾은 문자를 2번으로 줄이기

In [ ]:
def 반복문자_정규화(텍스트):
  return re.sub(r'(.)\1{2,}', r'\1\1', 텍스트)

텍스트A = "ㅋㅋㅋㅋㅋㅋ 너무 웃겨요"
결과A = 반복문자_정규화(텍스트A)

print("원본:", 텍스트A)
print("결과:", 결과A)
print()

텍스트B = "맛있어요ㅠㅠㅠㅠㅠ 최고야"
결과B = 반복문자_정규화(텍스트B)

print("원본:", 텍스트B)
print("결과:", 결과B)

원본: ㅋㅋㅋㅋㅋㅋ 너무 웃겨요
결과: ㅋㅋ 너무 웃겨요

원본: 맛있어요ㅠㅠㅠㅠㅠ 최고야
결과: 맛있어요ㅠㅠ 최고야


### 3) 어간 추출 및 표제어 추출
- **PorterStemmer**: 단어의 문법적인 뒤쪽 부분(접미사)를 규칙에 따라 제거 - 철자 기반
  - 한계: 과하게 꺾이거나(happily -> happ), 사전에 없는 단어 생성(happily -> happili)
- **WordNetLemmatizer**: 사전을 조회해서 원형 복원
  - 의미 보존이 중요할 때
  - 검색 및 정보 검색을 효율적으로
  - 데이터 정규화 (하나로 묶어 집합의 크기를 줄이기 위해)

In [ ]:
문장 = "그 영화는 정말 재미있었고 배우 연기도 훌륭했어요"

# stem 옵션 없이 (기본)
기본 = okt.morphs(문장)
print("stem=False (기본):", 기본)

# stem=True 적용
어간 = okt.morphs(문장, stem=True)      # okt.morphs: 한글 전용
print("stem=True (어간) :", 어간)
print()
print("→ '재미있었고' 가 '재미있다'로, '훌륭했어요'가 '훌륭하다'로 바뀝니다!")

stem=False (기본): ['그', '영화', '는', '정말', '재미있었고', '배우', '연기', '도', '훌륭했어요']
stem=True (어간) : ['그', '영화', '는', '정말', '재미있다', '배우', '연기', '도', '훌륭하다']

→ '재미있었고' 가 '재미있다'로, '훌륭했어요'가 '훌륭하다'로 바뀝니다!


In [ ]:
import nltk
nltk.download('wordnet')
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()
단어들 = [("running", "v"),("better", "a"),("mice", "n"),("went","v")]

for 단어, 품사 in 단어들:
  원형 = lemmatizer.lemmatize(단어, pos=품사)
  print(f"{단어:12} -> {원형}")   # f-string의 {:8} : 8칸 고정폭으로 출력 (표처럼 정렬)

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


running      -> run
better       -> good
mice         -> mouse
went         -> go


### 4. (4단계) 불용어 처리
불용어: 분석에 실질적으로 도움이 되지 않는 단어
- NLTK에는 이미 179개의 영어 불용어가 준비되어 있음

In [ ]:
from nltk.corpus import stopwords

영어_불용어 = stopwords.words("english")

print("불용어 개수:", len(영어_불용어))
print("예시:", 영어_불용어[:20])


불용어 개수: 198
예시: ['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been']


In [ ]:
from nltk.tokenize import word_tokenize

영어문장 = "This movie is really good and I enjoyed watching it very much"

# 1. 소문자로 통일 후 토큰화
토큰들 = word_tokenize(영어문장.lower())
print("토큰화 결과:", 토큰들)
print("단어 수:", len(토큰들), "개")
print()

토큰화 결과: ['this', 'movie', 'is', 'really', 'good', 'and', 'i', 'enjoyed', 'watching', 'it', 'very', 'much']
단어 수: 12 개



In [ ]:
# 2. 불용어 제거
# 리스트 컴프리헨션: 불용어 목록에 없는 단어만 남기기
핵심단어들 = [토큰 for 토큰 in 토큰들 if 토큰 not in 영어_불용어]
print("불용어 제거 후:", 핵심단어들)
print("단어 수:", len(핵심단어들), "개")
print()
print("→", len(토큰들), "개에서", len(핵심단어들), "개로! 핵심 단어만 남았습니다.")

불용어 제거 후: ['movie', 'really', 'good', 'enjoyed', 'watching', 'much']
단어 수: 6 개

→ 12 개에서 6 개로! 핵심 단어만 남았습니다.


한국어는 NLTK에서 제공하지 않아서 직접 만들어야 됨!!

In [ ]:
한국어_불용어 = [
    # 조사 (단어에 붙어서 문법적 의미를 추가하는 말)
    "이", "가", "은", "는", "을", "를", "의",
    "에", "에서", "로", "으로", "과", "와", "도", "만",
    # 접속사 (문장을 연결하는 말)
    "그리고", "그러나", "그런데", "그래서", "하지만",
    # 정도 부사 (의미는 있지만 분석 가치가 낮은 말)
    "정말", "진짜", "매우", "너무", "아주", "좀",
    # 기타 불필요 단어
    "것", "수", "있다", "없다", "합니다", "입니다"
]

print("한국어 불용어 개수:", len(한국어_불용어), "개")
print("목록:", 한국어_불용어)

한국어 불용어 개수: 32 개
목록: ['이', '가', '은', '는', '을', '를', '의', '에', '에서', '로', '으로', '과', '와', '도', '만', '그리고', '그러나', '그런데', '그래서', '하지만', '정말', '진짜', '매우', '너무', '아주', '좀', '것', '수', '있다', '없다', '합니다', '입니다']


In [ ]:
문장 = "이 음식은 정말 맛있고 가격도 매우 저렴합니다"

# 1. split()으로 토큰화
토큰들 = 문장.split()
print("토큰화:", 토큰들)
print()

# 2. 불용어 제거
결과 = [토큰 for 토큰 in 토큰들 if 토큰 not in 한국어_불용어]
print("불용어 제거 후:", 결과)

print()
print("--> 아직 '음식은', '가격도' 처럼 조사가 있다")
print("   형태소 분석(CH 7)을 먼저 하면 됨 (토큰화 하고 나서 불용어 토큰을 제거한거라)")

토큰화: ['이', '음식은', '정말', '맛있고', '가격도', '매우', '저렴합니다']

불용어 제거 후: ['음식은', '맛있고', '가격도', '저렴합니다']

--> 아직 '음식은', '가격도' 처럼 조사가 있다
   형태소 분석(CH 7)을 먼저 하면 됨 (토큰화 하고 나서 불용어 토큰을 제거한거라)


### 한국어 형태소 분석 - KoNLPy
**형태소**: 의미를 가진 가장 작은 언어 단위
KoNLPy는 한국어 형태소 분석을 해주는 python 라이브러리.

1. morphs(): 형태소 단위로 쪼개기
2. pos(): 품사 태깅(형태소 + 품사)
3. nouns(): 명사만 추출; 텍스트의 핵심 의미는 명사에 담겨 있는 경우가 많아서 많이 쓰임!
4. morphs(stem=True): 어간 추출

In [ ]:
문장 = "나는 오늘 학교에서 파이썬을 배웠어요"

# okt.morphs(): 형태소 분리
형태소들 = okt.morphs(문장)

print("원본: ", 문장)
print("형태소: ", 형태소들)

원본:  나는 오늘 학교에서 파이썬을 배웠어요
형태소:  ['나', '는', '오늘', '학교', '에서', '파이썬', '을', '배웠어요']


In [ ]:
# pos(): 품사 태깅 (형태소 + 품사)
# 각 형태소가 어떤 품사인지 함께 알려준다.

# okt.pos(): 각 형태소와 품사를 (단어, 품사) 쌍으로 변환

품사_결과 = okt.pos(문장)

print("원본:", 문장)
print("품사 태깅 결과:")
print(품사_결과)
print()

원본: 나는 오늘 학교에서 파이썬을 배웠어요
품사 태깅 결과:
[('나', 'Noun'), ('는', 'Josa'), ('오늘', 'Noun'), ('학교', 'Noun'), ('에서', 'Josa'), ('파이썬', 'Noun'), ('을', 'Josa'), ('배웠어요', 'Verb')]



In [ ]:
# ── pos() 결과를 표 형태로 예쁘게 출력하기 ────────

문장2 = "예쁜 카페에서 맛있는 커피를 마셨어요"

품사_설명 = {
    'Noun': '명사', 'Verb': '동사', 'Adjective': '형용사',
    'Adverb': '부사', 'Josa': '조사', 'Eomi': '어미',
    'Alpha': '영문자', 'Punctuation': '문장부호'
}

print(f"문장: {문장2}")
print()
# f-string의 {:8} : 8칸 고정폭으로 출력 (표처럼 정렬)
print(f"{'단어':10} {'품사코드':15} {'품사설명'}")
print("-" * 40)

for 단어, 품사 in okt.pos(문장2):
    # .get(키, 기본값) : 딕셔너리에서 키를 찾고, 없으면 기본값 반환
    설명 = 품사_설명.get(품사, 품사)
    print(f"  {단어:10} {품사:15} {설명}")

문장: 예쁜 카페에서 맛있는 커피를 마셨어요

단어         품사코드            품사설명
----------------------------------------
  예쁜         Adjective       형용사
  카페         Noun            명사
  에서         Josa            조사
  맛있는        Adjective       형용사
  커피         Noun            명사
  를          Josa            조사
  마셨어요       Verb            동사


In [ ]:
# nouns(): 명사만 추출 ---**가장 많이 쓰임**
문장3 = "오늘 배달 시킨 치킨이 정말 바삭하고 맛있었어요. 소스도 최고!"

# okt.nouns() : 명사만 추출해서 리스트로 반환
명사들 = okt.nouns(문장3)

print("원본:", 문장3)
print("명사:", 명사들)
print()
print("→ 핵심 단어만 딱 남습니다!")

원본: 오늘 배달 시킨 치킨이 정말 바삭하고 맛있었어요. 소스도 최고!
명사: ['오늘', '배달', '치킨', '정말', '바삭', '소스', '최고']

→ 핵심 단어만 딱 남습니다!


In [ ]:
# morphs(stem=True) : 어간 추출 (다양한 활용형을 원형으로 통일)

문장_A = "그 영화는 재미있었어요"   # 과거형
문장_B = "그 영화는 재미있어요"    # 현재형
문장_C = "그 영화는 재미있습니다"  # 격식체

print("[stem=False (기본)]")
print("  A:", okt.morphs(문장_A))
print("  B:", okt.morphs(문장_B))
print("  C:", okt.morphs(문장_C))
print()

print("[stem=True (원형으로 통일)]")
print("  A:", okt.morphs(문장_A, stem=True))
print("  B:", okt.morphs(문장_B, stem=True))
print("  C:", okt.morphs(문장_C, stem=True))

[stem=False (기본)]
  A: ['그', '영화', '는', '재미있었어요']
  B: ['그', '영화', '는', '재미있어요']
  C: ['그', '영화', '는', '재미있습니다']

[stem=True (원형으로 통일)]
  A: ['그', '영화', '는', '재미있다']
  B: ['그', '영화', '는', '재미있다']
  C: ['그', '영화', '는', '재미있다']


### 파이프라인 통합 실습

In [ ]:
불용어_목록 = [
    "것", "수", "나", "저", "제", "그", "이", "때",
    "등", "좀", "잘", "더", "한", "안", "못", "또"
]

def 전처리_파이프라인(텍스트):
  """
  입력: 원본 텍스트 (문자열)
  출력: 전처리된 핵심 단어 리스트
  """
  # 1단계: 정제 - 노이즈 제거
  텍스트 = re.sub(r'http\S+', '', 텍스트)              # URL 제거
  텍스트 = re.sub(r'<[^>]+>', '', 텍스트)              # HTML 태그 제거
  텍스트 = re.sub(r'[^가-힣a-zA-Z0-9 ]', '', 텍스트)    # 특수문자 제거
  텍스트 = re.sub(r'\s+', ' ', 텍스트).strip()         # 공백 정리

  # 2단계: 정규화 - 반복 문자 압축
  텍스트 = re.sub(r'(.)\1{2,}', r'\1\1', 텍스트)

  # 3단계: 형태소 분석 - 명사 추출
  토큰들 = okt.nouns(텍스트)

  # 4단계: 불용어 제거 + 2글지 이상만 유지
  결과 = [단어 for 단어 in 토큰들 if 단어 not in 불용어_목록 and len(단어) >= 2]

  return 결과

print("파이프라인 함수 준비 완료")

파이프라인 함수 준비 완료


In [ ]:
# ── 파이프라인 테스트 1 ──────────────────────

리뷰1 = "ㅋㅋㅋ 진짜 맛있어요!!!! http://naver.com 치킨 배달 강추★★★"
결과1 = 전처리_파이프라인(리뷰1)

print("원본:", 리뷰1)
print("결과:", 결과1)
print("---")

# ── 파이프라인 테스트 2 ──────────────────────

리뷰2 = "배송이 너무 느려서 음식이 다 식었어요ㅠㅠㅠ 다시는 안 시킬듯"
결과2 = 전처리_파이프라인(리뷰2)

print("원본:", 리뷰2)
print("결과:", 결과2)
print("---")

# ── 파이프라인 테스트 3 ──────────────────────

리뷰3 = "가격 대비 양도 많고 맛도 괜찮았습니다. 재주문 의사 100% 있습니다."
결과3 = 전처리_파이프라인(리뷰3)

print("원본:", 리뷰3)
print("결과:", 결과3)
print("---")

# ── 파이프라인 테스트 4 ──────────────────────

리뷰4 = "포장이 예쁘고 서비스도 훌륭해요!! 친절한 사장님 덕분에 기분 좋았어요"
결과4 = 전처리_파이프라인(리뷰4)

print("원본:", 리뷰4)
print("결과:", 결과4)
print("---")

# ── 파이프라인 테스트 5 ──────────────────────

리뷰5 = "솔직히 기대보다 별로였어요. 홍보랑 실제 음식이 많이 달랐네요."
결과5 = 전처리_파이프라인(리뷰5)

print("원본:", 리뷰5)
print("결과:", 결과5)
print("---")

원본: ㅋㅋㅋ 진짜 맛있어요!!!! http://naver.com 치킨 배달 강추★★★
결과: ['진짜', '치킨', '배달', '강추']
---
원본: 배송이 너무 느려서 음식이 다 식었어요ㅠㅠㅠ 다시는 안 시킬듯
결과: ['배송', '음식']
---
원본: 가격 대비 양도 많고 맛도 괜찮았습니다. 재주문 의사 100% 있습니다.
결과: ['가격', '대비', '양도', '주문', '의사']
---
원본: 포장이 예쁘고 서비스도 훌륭해요!! 친절한 사장님 덕분에 기분 좋았어요
결과: ['포장', '서비스', '사장', '덕분', '기분']
---
원본: 솔직히 기대보다 별로였어요. 홍보랑 실제 음식이 많이 달랐네요.
결과: ['기대', '별로', '홍보', '실제', '음식']
---


In [ ]:
# ── 전처리 결과로 단어 빈도 분석 ────────────────
# 5개 리뷰의 단어를 합쳐서 가장 많이 나온 단어를 찾습니다

# extend() : 리스트에 다른 리스트의 항목을 모두 추가
모든_단어 = []
모든_단어.extend(결과1)  # 리뷰1 단어 추가
모든_단어.extend(결과2)  # 리뷰2 단어 추가
모든_단어.extend(결과3)  # 리뷰3 단어 추가
모든_단어.extend(결과4)  # 리뷰4 단어 추가
모든_단어.extend(결과5)  # 리뷰5 단어 추가

In [ ]:
# ── TOP 10 단어 출력 (막대 그래프 포함) ──────────

# Counter() : 각 단어가 몇 번 나왔는지 세어줍니다
빈도 = Counter(모든_단어)

print("단어 빈도 TOP 10")
print()
print(f"{'순위':4} {'단어':10} {'횟수':5}  막대")
print("-" * 35)

# .most_common(10) : 가장 많이 나온 순서로 10개 반환
for 순위, (단어, 횟수) in enumerate(빈도.most_common(10), 1):
    막대 = "█" * 횟수  # 횟수만큼 블록 기호 반복
    print(f"  {순위:2}위  {단어:10} {횟수:3}회  {막대}")


단어 빈도 TOP 10

순위   단어         횟수     막대
-----------------------------------
   1위  음식           2회  ██
   2위  진짜           1회  █
   3위  치킨           1회  █
   4위  배달           1회  █
   5위  강추           1회  █
   6위  배송           1회  █
   7위  가격           1회  █
   8위  대비           1회  █
   9위  양도           1회  █
  10위  주문           1회  █


## 복습 과제
**목표:**
1. 영화 리뷰 5개를 파이프라인으로 전처리
2. 단어를 모두 합쳐서 빈도 분석
3. TOP 5 단어 출력

- 영화리뷰 분석 10개 (파이프라인 함수로)
- 결론 및 한계점

In [ ]:
def text_prepro(review):
  """
  텍스트 전처리 함수: 정제 -> 정규화 -> 형태소분석 -> 불용어제거
  """
  stopwords = {
            "은", "는", "이", "가", "을", "를", "에", "의", "도", "로", "으로",
            "와", "과", "한", "하다", "해서", "했다", "하고", "것", "수", "좀",
            "진짜", "너무", "꼭", "함", "됨", "영화"
  }

  # 1. 정제
  review = re.sub(r"http\S+|www\S+", " ", review)
  review = re.sub(r"<[^>]+>", " ", review)
  review = re.sub(r"[^가-힣a-zA-Z0-9 ]", " ", review)
  review = re.sub(r"\s+", " ", review).strip()

  # 2. 정규화
  review = re.sub(r'(.)\1{2,}', r'\1\1', review)

  # 3. 형태소 분석
  tokens = okt.morphs(review, stem=True)

  # 4. 불용어 제거
  review = [tok for tok in tokens if tok not in stopwords and len(tok)>1]

  return review

In [ ]:
# 영화 <왕과 사는 남자>

reviews = [
    "연출보다는 연기력이 좋은 영화, 그 연기력을 끌어올리는 감독도 능력",
    "방금 1200만 돌파했다는 왕과사는남자 👏👏",
    "두 남자의 눈빛이 다했다",
    "무조건 봐야 하는 영화. 영화가 끝나서도 가슴 먹먹한 슬픔",
    "이거는 꼭 봐야 함. 단전에서 울리는 감동…",
    "감히 2026년 최고의 영화라고 생각합니다! 단종 눈빛만 봐도 눈물이 ㅠㅠ",
    "최고의 영화, 단종의 재발견! 꼭 봐야 됨.",
    "한 줄 역사를 풀어낸 감독의 눈썰미 칭찬해",
    "와 진짜 ㅈㄴ재밋음.. ㅠㅠ",
    "유쾌하게 시작했다가 엄청난 여운으로 끝나는 영화ㅠㅠ오랜만에 웰메이드 사극이라 꼭 극장가서 보는 거 추천합니다!"
    ]

for idx, i in enumerate(reviews):
  print(f"리뷰{idx+1}: {i}")

print()

processed_reviews = [text_prepro(review) for review in reviews]

for i, review in enumerate(processed_reviews, 1):
  print(f"리뷰{i}: {review}")

리뷰1: 연출보다는 연기력이 좋은 영화, 그 연기력을 끌어올리는 감독도 능력
리뷰2: 방금 1200만 돌파했다는 왕과사는남자 👏👏
리뷰3: 두 남자의 눈빛이 다했다
리뷰4: 무조건 봐야 하는 영화. 영화가 끝나서도 가슴 먹먹한 슬픔
리뷰5: 이거는 꼭 봐야 함. 단전에서 울리는 감동…
리뷰6: 감히 2026년 최고의 영화라고 생각합니다! 단종 눈빛만 봐도 눈물이 ㅠㅠ
리뷰7: 최고의 영화, 단종의 재발견! 꼭 봐야 됨.
리뷰8: 한 줄 역사를 풀어낸 감독의 눈썰미 칭찬해
리뷰9: 와 진짜 ㅈㄴ재밋음.. ㅠㅠ
리뷰10: 유쾌하게 시작했다가 엄청난 여운으로 끝나는 영화ㅠㅠ오랜만에 웰메이드 사극이라 꼭 극장가서 보는 거 추천합니다!

리뷰1: ['연출', '보다는', '연기력', '좋다', '연기력', '끌다', '올리다', '감독', '능력']
리뷰2: ['방금', '1200만', '돌파', '살다', '남자']
리뷰3: ['남자', '눈빛']
리뷰4: ['무조건', '보다', '끝나다', '가슴', '먹다', '먹다', '슬픔']
리뷰5: ['보다', '단전', '에서', '울리다', '감동']
리뷰6: ['감히', '2026년', '최고', '라고', '생각', '단종', '눈빛', '보다', '눈물']
리뷰7: ['최고', '단종', '발견', '보다', '되다']
리뷰8: ['역사', '풀다', '내다', '감독', '눈썰미', '칭찬']
리뷰9: ['오다', '재밋음']
리뷰10: ['유쾌하다', '시작', '엄청나다', '여운', '끝나다', '오랜', '메이드', '사극', '이라', '장가', '보다', '추천']


In [ ]:
all_reviews = []
for review in processed_reviews:
  all_reviews.extend(review)

print("전체 단어 목록:", all_reviews)
print("전체 단어 수:", len(all_reviews))

전체 단어 목록: ['연출', '보다는', '연기력', '좋다', '연기력', '끌다', '올리다', '감독', '능력', '방금', '1200만', '돌파', '살다', '남자', '남자', '눈빛', '무조건', '보다', '끝나다', '가슴', '먹다', '먹다', '슬픔', '보다', '단전', '에서', '울리다', '감동', '감히', '2026년', '최고', '라고', '생각', '단종', '눈빛', '보다', '눈물', '최고', '단종', '발견', '보다', '되다', '역사', '풀다', '내다', '감독', '눈썰미', '칭찬', '오다', '재밋음', '유쾌하다', '시작', '엄청나다', '여운', '끝나다', '오랜', '메이드', '사극', '이라', '장가', '보다', '추천']
전체 단어 수: 62


In [ ]:
freq = Counter(all_reviews)

print("단어 빈도 TOP 10")
print()
print(f"{'rank':4} {'word':10} {'cnt':5}  bar")
print("-" * 35)
for rank, (word, cnt) in enumerate(freq.most_common(5),1):
  bar = "█" * cnt
  print(f" {rank:2}위  {word:10} {cnt:3}회   {bar}")

단어 빈도 TOP 10

rank word       cnt    bar
-----------------------------------
  1위  보다           5회   █████
  2위  연기력          2회   ██
  3위  감독           2회   ██
  4위  남자           2회   ██
  5위  눈빛           2회   ██
